In [1]:
import pandas as pd
import re

In [5]:
# defining input and output files 
input="credit_txn_v5.xlsx"
output="output_file.xlsx"


In [6]:
df=pd.read_excel(input)

In [8]:
# Grouping transactions by group name 
# Counting frequency of Actual Ledger Name and sorting them in decreasing order

tables = {
    a: (
        group["Actual Ledger Name"]
        .value_counts()
        .reset_index(name="frequency")
        .rename(columns={"index": "Actual Ledger Name"})
        .sort_values(by="frequency", ascending=False)
        .reset_index(drop=True)
    )
    for a, group in df.groupby("Direct Group")
}


In [ ]:
# exporting tables in excel file with sheet-names as group 

output_file = "ledger_freq_group-wise_sheetname_as_group.xlsx"
used_sheet_names = set()

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    for a, table in tables.items():
        sheet_name = str(a)
        sheet_name = re.sub(r'[\\/*?:\[\]]', '', sheet_name).strip()

        if not sheet_name:
            sheet_name = "Group"

        sheet_name = sheet_name[:31]

        original = sheet_name
        i = 1
        while sheet_name in used_sheet_names:
            suffix = f"_{i}"
            sheet_name = original[:31 - len(suffix)] + suffix
            i += 1

        used_sheet_names.add(sheet_name)
        table.to_excel(writer, sheet_name=sheet_name, index=False)

In [ ]:
# Counting ledgers frequency company wise 

ledger_freq = (
    df
    .groupby(
        ['Company ID', 'Company Name', 'Actual Ledger Name'],
        as_index=False
    )
    .size()
    .rename(columns={'size': 'frequency'})
    .sort_values(
        ['Company ID', 'frequency'],
        ascending=[True, False]
    )
)


In [ ]:
# counting group frequency company-wise
group_freq = (
    df
    .groupby(
        ['Company ID', 'Company Name', 'Group'],
        as_index=False
    )
    .size()
    .rename(columns={'size': 'frequency'})
    .sort_values(
        ['Company ID', 'frequency'],
        ascending=[True, False]
    )
)


In [ ]:
# finding ledgers whose frequency is greater than average frequency 

ledgers_above_avg = (
    ledger_freq
    .assign(
        avg_freq=ledger_freq.groupby('Company ID')['frequency'].transform('mean')
    )
    .query('frequency > avg_freq')
)


In [ ]:
# sorting ledger freq data with company name and then frequency
# exporting sorted data
export_df = (
    ledger_freq[
        [
            'Company Name',
            'Company ID',
            'Actual Ledger Name',
            'frequency'
        ]
    ]
    .sort_values(
        by=['Company Name', 'frequency'],
        ascending=[True, False]
    )
)


export_df.to_excel(
    "ledger_freq_company-wise.xlsx",
    index=False
)

In [ ]:
# Filtering data for Group-wise ledger frequency and exporting the data in sorted manner 

final_df = pd.concat(
    [
        table.assign(group=a)
        for a, table in tables.items()
    ],
    ignore_index=True
)

final_df = (
    final_df[
        ['group', 'Actual Ledger Name', 'frequency']
    ]
    .rename(columns={'frequency': 'freq'})
    .sort_values(
        by=['group', 'freq'],
        ascending=[True, False]
    )
)

final_df.to_excel(
    "ledger_freq_group-wise.xlsx",
    sheet_name="Group_Ledger_Frequency",
    index=False
)

